# NB01 — Phase 1: build M_Q and prove it is the same function

**Budget: 3.5 h**

Needs a single-GPU 80 GB pod for the 8B pass. 
The gate needs an unrotated reference and `M_Q` at the same time, so ~32 GB of weights plus logits, so a 48 GB pod (A40, L40S, A6000) will also work. 
The debug path runs on anything.

We do an orthogonal transform $Q$ of $M$'s residual stream leaving the RMSNorm transformer's function the same while sending every extracted activation to $Qv$. 
This behavioral similarity between explainer and target is untouched, but coordinate alignment is now random. 
Anything that survives the rotation is not coordinate-frame.

This notebook produces $M_Q$ and proves it computes the same function as $M$.

Deliverables: `rotate.py` (already in `se/`), $Q$ + its seed checkpointed, and the gate
numbers written where the write-up can quote them.


In [ ]:
# --- dependencies -----------------------------------------------------------
# A RunPod image ships torch built against the pod's own driver, so nothing here may
# replace it: se_env pins the installed torch as a pip constraint and installs only what
# is missing or below the floor these notebooks need — including bitsandbytes, which Colab
# had preinstalled and RunPod images do not (se_common asks for paged_adamw_8bit).
# Safe to re-run: a no-op on a warm pod.
import os
import sys

# `se/` holds the shared modules: se_env, se_config, rotate, se_common. Clone this repo
# onto the pod's volume (/workspace/self_explainer) so it survives the pod.
REPO_DIR = os.environ.get("SE_REPO_DIR", "/workspace/self_explainer")
if not os.path.isdir(os.path.join(REPO_DIR, "se")):
    REPO_DIR = os.path.abspath(".." if os.path.isdir("../se") else ".")
sys.path.insert(0, os.path.join(REPO_DIR, "se"))

import se_env

se_env.ensure_deps()


In [ ]:
# --- environment ------------------------------------------------------------
# Caches and outputs go on the pod's volume, never the container disk: /workspace is what
# survives a stopped or terminated pod, and the 8B checkpoint alone is 16 GB. HF_TOKEN comes
# from the pod template's environment or <volume>/.hf_token — there is no prompt to answer,
# because a preempted pod restarts with nobody watching.
#
# The gate holds two 8B bf16 copies at once (~32 GB resident), so 48 GB is the floor
# and 80 GB is comfortable.
env = se_env.bootstrap(repo_dir=REPO_DIR, gpu="required", min_vram_gb=48)

import se_config as C


## 0. Cheap gate first

`se/test_rotate.py` runs the whole construction on a 3-layer model in float64 on CPU to catch any errors.


In [ ]:
import subprocess

print(subprocess.run([sys.executable, f"{REPO_DIR}/se/test_rotate.py"],
                     capture_output=True, text=True).stdout)


## 0b. Debug on Qwen3-0.6B first

0.6B is a fast alternative for debugging the fold and the gate. 

If it passes on 0.6B and fails 8B then the problem is not the math but scale or memory.

**Run this in float32.** `M = M_Q` is exact algebra, so the gate only ever measures arithmetic
error, and bfloat16 carries 8 mantissa bits. `apply_rotation` mixes all `d` coordinates and then
re-rounds every weight, which on this model puts mean KL at 1.2e-3 against a 1e-4 bar — a
failure that says nothing about the construction. v2 §5.5 predicted it. Same construction on the
mini replica: bf16 `1.3e-05`, fp16 `2.0e-07`, fp32 `7.4e-10`, fp64 `0`. 0.6B in fp32 is ~5 GB
for both copies. This cell is where correctness is established; the 8B pass stays in bf16 for
memory and is judged against its own quantization floor instead.


In [ ]:
DEBUG_MODEL_ID = "Qwen/Qwen3-0.6B"
RUN_DEBUG_PASS = True

if RUN_DEBUG_PASS:
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer

    import rotate as R

    # float32, not bfloat16, and that is the point of this cell. M = M_Q is an identity in
    # exact arithmetic, so everything the gate measures is arithmetic error — and bf16 has 8
    # mantissa bits. apply_rotation computes W Q^T in float64 and stores it back with
    # .to(w.dtype), so in bf16 every weight is re-rounded to ~3 decimal digits AFTER a dense
    # mix of all d coordinates. On this model that alone puts mean KL at 1.2e-3 against a 1e-4
    # bar: a failure that says nothing about whether the construction is right. v2 §5.5
    # predicted it — "a fixed 1e-4 threshold may be unreachable for reasons that are not bugs."
    #
    # So the demonstration runs where the claim can hold. Same construction on the mini_qwen3
    # replica, fold+rotate: bf16 1.3e-05, fp16 2.0e-07, fp32 7.4e-10, fp64 0. 0.6B in fp32 is
    # 2.4 GB, two copies ~5 GB, which fits anywhere. The 8B pass below stays in bf16 for memory
    # and is gated against its own quantization floor instead.
    DEBUG_DTYPE = torch.float32

    dbg_tok = AutoTokenizer.from_pretrained(DEBUG_MODEL_ID)
    if dbg_tok.pad_token is None:
        dbg_tok.pad_token = dbg_tok.eos_token

    dbg_ref = AutoModelForCausalLM.from_pretrained(
        DEBUG_MODEL_ID, dtype=DEBUG_DTYPE, device_map="auto").eval()
    dbg = AutoModelForCausalLM.from_pretrained(
        DEBUG_MODEL_ID, dtype=DEBUG_DTYPE, device_map="auto").eval()

    print(f"{DEBUG_MODEL_ID}: d = {dbg.config.hidden_size}, "
          f"tied = {R.tied_embeddings(dbg)}, dtype = {DEBUG_DTYPE}")
    if R.tied_embeddings(dbg):
        R.untie_embeddings(dbg)
        print("untied lm_head (folding would otherwise corrupt the embedding)")

    R.fold_rmsnorm_gains(dbg)
    dbg_Q = R.random_orthogonal(dbg.config.hidden_size, seed=C.Q_SEED)
    R.apply_rotation(dbg, dbg_Q)

    dbg_texts = ["The capital of France is Paris, and the capital of Germany is Berlin. " * 8,
                 "In a shocking finding, scientists discovered a herd of unicorns. " * 8,
                 "def fibonacci(n):\n    if n < 2:\n        return n\n" * 4,
                 "The mitochondrion is the powerhouse of the cell. " * 8]
    dbg_report = R.invariance_gate(dbg_ref, dbg, dbg_tok, dbg_texts,
                                   max_length=128, batch_size=2)
    print()
    print(R.format_gate_report(dbg_report, f"DEBUG GATE — {DEBUG_MODEL_ID} (float32)"))
    assert dbg_report["passed"], (
        "fix the construction here before loading 8B. In float32 the absolute 1e-4 bar is "
        "reachable with ~5 orders to spare, so a failure here is a real bug — a missed gain "
        "fold, a transposed [vocab, d] convention, or an unnoticed tied embedding."
    )

    del dbg_ref, dbg
    import gc
    gc.collect()
    torch.cuda.empty_cache()


## 1. Load the target


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

import rotate as R

MODEL_ID = C.TARGET_MODEL_ID

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, dtype=torch.bfloat16, device_map="auto",
).eval()

d = model.config.hidden_size
print(f"{MODEL_ID}: d = {d}, layers = {model.config.num_hidden_layers}")
print(f"tied embeddings: {R.tied_embeddings(model)}")
assert not R.tied_embeddings(model), "untie first — folding would corrupt the embedding"


## 2. Evaluation texts

~256 FineWeb sequences at 512 tokens. 
These are also the texts NB05 fits the ridge projector on, so they are cached to disk and reused.


In [ ]:
import json

from datasets import load_dataset

GATE_TEXTS_PATH = f"{C.ROTATION_DIR}/gate_texts.json"
N_GATE_SEQS = 256
GATE_MAX_LEN = 512

if os.path.exists(GATE_TEXTS_PATH):
    with open(GATE_TEXTS_PATH) as f:
        gate_texts = json.load(f)
    print(f"loaded {len(gate_texts)} cached gate texts")
else:
    fw = load_dataset("HuggingFaceFW/fineweb", name="sample-10BT",
                      split="train", streaming=True)
    gate_texts = []
    for row in fw:
        if len(row["text"]) > 2000:            # long enough to fill 512 tokens
            gate_texts.append(row["text"])
        if len(gate_texts) >= N_GATE_SEQS:
            break
    with open(GATE_TEXTS_PATH, "w") as f:
        json.dump(gate_texts, f)
    print(f"cached {len(gate_texts)} gate texts to {GATE_TEXTS_PATH}")


## 3. Fold the RMSNorm gains, and verify folding alone

$\operatorname{RMSNorm}(x) = x/\operatorname{rms}(x) \odot g$. 
The elementwise gain doesn't commute with $Q$, so we absorb it into every linear map that takes normalized output:

- `input_layernorm` → `q_proj`, `k_proj`, `v_proj`
- `post_attention_layernorm` → `gate_proj`, `up_proj`
- final `model.norm` → `lm_head`

Qwen3's `q_norm`/`k_norm` are deliberately untouched: they normalize head-dimension slices
downstream of the read matrix, inside the head, so a residual-stream rotation doesn't reach
them. 
RoPE likewise operates post-projection.

After folding, normalization is pure $x/\operatorname{rms}(x)$, which commutes with any orthogonal $Q$.


In [ ]:
import time

t0 = time.time()
folded = R.fold_rmsnorm_gains(model)     # in place
R.assert_no_residual_gains(folded)
print(f"folded in {time.time() - t0:.1f}s; all residual-stream gains are now 1")


In [ ]:
# Gate A: folding alone. This is the first check AND this notebook's quantization floor.
#
# Folding is an exact identity in exact arithmetic — it moves each gain out of the norm and
# into the linear map that consumes it — so every deviation below is bf16 re-rounding with no
# change of basis involved. That number is what rewriting every weight costs at this dtype,
# which makes it the right scale to judge the rotation against (§5 below).
reference = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, dtype=torch.bfloat16, device_map="auto",
).eval()

fold_report = R.invariance_gate(reference, folded, tokenizer, gate_texts[:64],
                                max_length=GATE_MAX_LEN, batch_size=2)
print(R.format_gate_report(fold_report, "GATE A — folding only (M vs M_folded) = the floor",
                           show_verdict=False))

print("\nRead this as a measurement, not a pass/fail against 1e-4: in bf16 a nonzero KL here")
print("is arithmetic, not a bug. The absolute bar is enforced in float32, on 0.6B, in §0b.")
print("The check that folding did not break anything is the next cell, not this one.")

### Gate A's verdict — on decided tokens, not on the aggregate

The top-1 agreement Gate A prints is not a floor-free quantity, so it cannot be asserted on
directly. Re-rounding perturbs each logit by a bounded amount, so it can only flip a
prediction whose top-1/top-2 gap was already inside that amount. How many tokens that is
depends entirely on how many near-ties the *reference* had, which is a property of the text,
not of the transform — a percent or two on FineWeb prose, essentially none on repetitive text.
An aggregate agreement anywhere in the 0.97–0.99 band is therefore consistent with both a
correct fold and no fold at all, and a fixed bar across that band fails correct constructions.

Resolving the flips by that gap does separate the two, and by a wide margin. Measured on
Qwen3-0.6B in bf16 over non-repetitive prose, `se/rotate.py` unchanged:

| arm | mean KL | top-1 | agreement on gap > 1 |
|---|---|---|---|
| fold — the real transform | 1.6e-03 | 0.9776 | **1.00000** |
| re-round identity: `w ← round(round(w·g)/g)`, no fold, no basis change | 1.5e-03 | 0.9790 | **1.00000** |
| fold + rotate | 2.6e-03 | 0.9748 | **1.00000** |
| bug: `model.norm` gain cleared instead of absorbed | 7.5e+00 | 0.1066 | 0.16486 |
| bug: `v_proj` missed by the `input_layernorm` fold | 8.7e+00 | 0.0098 | 0.01351 |

The re-round arm is the control that settles it: it changes nothing about the model — it
multiplies by each gain and divides straight back out, leaving the norms alone — and it lands
on the same aggregate agreement as the fold. Whatever the aggregate is measuring, it is not
the fold.

A decided token is out of reach of rounding and is flipped by a missed gain almost always, so
that is where the assertion goes. `max_kl` above is the independent confirmation: a token
whose top-1 changed at a gap of 1 nat has had its distribution rearranged by O(0.1–1) nats of
KL, so a `max_kl` in the hundredths bounds the damage across every token at once.

In [ ]:
# The assertion Gate A's numbers cannot carry on their own (see above).
fold_profile = R.disagreement_profile(reference, folded, tokenizer, gate_texts[:64],
                                      max_length=GATE_MAX_LEN, batch_size=2)
print(R.format_disagreement_profile(
    fold_profile, "GATE A — where the top-1 flips sit (M vs M_folded)"))

# Read the bins: a correct fold puts every flip in the two leftmost ones and leaves the right
# tail untouched. A missed gain flips at a roughly flat rate all the way across.
assert fold_profile["n_confident_tokens"] > 1000, "no decided tokens — check the texts"
assert fold_profile["confident_agreement"] > 0.999 and fold_report["mean_kl"] < 1e-2, (
    "folding broke the model — the flips are not confined to near-ties, which bf16 "
    "re-rounding cannot do. Fix before touching Q."
)

## 4. Build and apply Q

Haar-uniform $Q ∈ O(d)$ via QR of a float64 Gaussian. We fix and record the seed.

Write paths see $W \leftarrow QW$, $W \leftarrow WQ^T$ and the `[vocab, d]` tensors get $X \leftarrow XQ^T$.


In [ ]:
Q = R.random_orthogonal(d, seed=C.Q_SEED)
orth_err = (Q.T @ Q - torch.eye(d, dtype=torch.float64)).abs().max().item()
print(f"Q: {tuple(Q.shape)}, seed = {C.Q_SEED}, max |Q^T Q - I| = {orth_err:.2e}")

q_path = R.save_rotation(Q, C.Q_SEED, f"{C.ROTATION_DIR}/Q_seed{C.Q_SEED}.pt")
print(f"checkpointed to {q_path}")


In [ ]:
t0 = time.time()
rotated = R.apply_rotation(folded, Q)       # in place; `folded` is now M_Q
print(f"rotated in {time.time() - t0:.1f}s")


## 5. Gate B — the invariance check

**Judged against Gate A's floor, not against an absolute 1e-4.** Folding and rotating are both
exact identities in exact arithmetic, and both re-round every weight to bf16 — but only the
rotation also changes the basis. So Gate A measures what the re-rounding costs, and Gate B asks
what the *rotation* added on top of it. The bar is `max(1e-4, 3x floor)`: it never loosens below
the absolute one, and it reduces to it exactly when the floor is small, which is why §0b's
float32 pass is still where correctness is established.

The relative form keeps the power the absolute one had. The failures this is meant to catch — a
missed gain fold, a transposed `[vocab, d]` convention, an unnoticed tied embedding — land
orders above the floor, not a few multiples of it. A perturbed model measures ~120x.


In [ ]:
gate_report_raw = R.invariance_gate(reference, rotated, tokenizer, gate_texts,
                                    max_length=GATE_MAX_LEN, batch_size=2)

# Judged against Gate A's floor rather than the absolute 1e-4. Both transforms are exact in
# exact arithmetic and both are re-rounded to bf16; only this one also changes the basis, so
# the floor isolates what the rotation actually added. The bar is max(1e-4, 3x floor), so it
# never loosens below the absolute one and reduces to it exactly when the floor is small —
# which is why the float32 debug pass in §0b is still the test of correctness.
gate_report = R.gate_against_floor(gate_report_raw, fold_report,
                                   kl_ratio=3.0, top1_ratio=3.0)
print(R.format_gate_report(gate_report, "GATE B — full transform (M vs M_Q)"))

# And the same gap-resolved view as Gate A, on the same subset, so the two are comparable.
# This is the part of the gate that a ratio-to-floor cannot express: it asks not "how much
# bigger than the floor" but "is the damage still confined to tokens the reference had not
# decided", which is the actual claim being made about the rotation.
rot_profile = R.disagreement_profile(reference, rotated, tokenizer, gate_texts[:64],
                                     max_length=GATE_MAX_LEN, batch_size=2)
print()
print(R.format_disagreement_profile(
    rot_profile, "GATE B — where the top-1 flips sit (M vs M_Q)"))

gate_report["q_seed"] = C.Q_SEED
gate_report["dtype"] = "bfloat16"
gate_report["model"] = MODEL_ID
with open(f"{C.REPORTS_DIR}/invariance_gate.json", "w") as f:
    json.dump({"fold_only": fold_report,
               "fold_only_profile": fold_profile,
               "fold_and_rotate": gate_report,
               "fold_and_rotate_profile": rot_profile,
               "fold_and_rotate_absolute_bar": gate_report_raw,
               "debug_float32": globals().get("dbg_report")}, f, indent=2)
print(f"\nwritten to {C.REPORTS_DIR}/invariance_gate.json — quote these in the write-up (§11.6)")

assert gate_report["passed"], (
    "INVARIANCE GATE FAILED — do not proceed to NB02. The rotation cost more than 3x the "
    "quantization floor, which bf16 re-rounding alone does not explain."
)
assert rot_profile["n_confident_tokens"] > 1000, "no decided tokens — check the texts"
assert rot_profile["confident_agreement"] > 0.999, (
    "INVARIANCE GATE FAILED — the rotation flipped predictions the reference had already "
    "decided. That is a construction error, not arithmetic. Do not proceed to NB02."
)

## 6. Second check — labels through the hook path

Recompute has-changed labels for ~200 patching examples under $M_Q$ and confirm they match the cache. 
This validates the transform through the hook path the experiment will use: $Qv$ patched into $M_Q$ rather than $v$ into $M$.

Field names come from the schema NB00 printed. 
If `resolve_position` raises, read the message and add the key the dataset schema.


In [ ]:
POSITION_KEYS = ("position", "index", "idx", "token_idx", "token_index", "pos",
                 "orig_text_token_idx", "token_position")


def resolve_position(patch_position):
    for k in POSITION_KEYS:
        if k in patch_position and isinstance(patch_position[k], int):
            return patch_position[k]
    raise KeyError(
        "Could not find the patched token index in patch_position. "
        f"Available keys: {sorted(patch_position.keys())}. "
        "Add the right one to POSITION_KEYS."
    )


def patch_hook(vector, position):
    """Replace the residual stream at `position` with `vector`, at each hooked layer."""
    def hook(module, args, output):
        hidden = output[0] if isinstance(output, tuple) else output
        hidden[:, position, :] = vector.to(hidden.dtype)
        return (hidden,) + output[1:] if isinstance(output, tuple) else hidden
    return hook


@torch.no_grad()
def replay_patch(model, example, vector, n_new_tokens):
    """Run the cached patch through `model` and return the greedy continuation."""
    layers = example["layer"]
    position = resolve_position(example["patch_position"])
    input_ids = torch.tensor(
        [tokenizer.convert_tokens_to_ids(example["input_tokens"])], device=model.device
    )
    v = torch.tensor(vector, device=model.device)

    handles = [model.model.layers[l].register_forward_hook(patch_hook(v, position))
               for l in layers]
    try:
        out = model.generate(input_ids, max_new_tokens=n_new_tokens, do_sample=False,
                             pad_token_id=tokenizer.pad_token_id)
    finally:
        for h in handles:
            h.remove()
    return tokenizer.convert_ids_to_tokens(out[0, input_ids.shape[1]:])


In [ ]:
N_LABEL_CHECK = 200

label_ds = load_dataset(C.ACT_DATASET, split=f"train[:{N_LABEL_CHECK}]")
agree, checked, failures = 0, 0, []

for i, ex in enumerate(label_ds):
    try:
        v = torch.tensor(ex["patch_position"]["intervention_vector"], dtype=torch.float64)
        Qv = (Q @ v).to(torch.bfloat16)
        n_new = len(ex["ablated_continuation"])
        got = replay_patch(rotated, ex, Qv, n_new)
    except KeyError as err:
        print(err)
        break
    except Exception as err:                       # noqa: BLE001 — log and continue
        failures.append((i, repr(err)))
        continue

    cached = ex["ablated_continuation"]
    checked += 1
    agree += int("".join(got) == "".join(cached))

if checked:
    print(f"replayed {checked} examples under M_Q with Qv patched in")
    print(f"  continuation matches cache : {agree}/{checked} = {agree/checked:.3f}")
    print(f"  errors                     : {len(failures)}")
    with open(f"{C.REPORTS_DIR}/label_recheck.json", "w") as f:
        json.dump({"checked": checked, "agree": agree, "rate": agree / checked,
                   "errors": failures[:20]}, f, indent=2)


## 7. Next steps

$M_Q$ is not saved. 
Nothing downstream needs it: training and evaluation consume cached activations, and the only thing that has to persist is $Q$ itself, which is small and reproducible from its seed. 

Next: **NB02** applies $v \to Qv$ to the cached activation dataset.


In [ ]:
print(f"Q            : {q_path}  (seed {C.Q_SEED})")
print(f"gate report  : {C.REPORTS_DIR}/invariance_gate.json")
print(f"gate passed  : {gate_report['passed']}")
print(f"gate texts   : {GATE_TEXTS_PATH}  (reused by NB05's ridge fit)")

del reference, rotated, folded, model
import gc
gc.collect()
torch.cuda.empty_cache()
